In [5]:
import os
import csv
import psycopg2

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [106]:
import pandas as pd
import numpy as np
import uuid
import random
from datetime import datetime, timedelta

np.random.seed(42)


# ------------------------
# CONFIG
# ------------------------
NUM_USERS = 2000
NUM_ISSUERS = 200
NUM_TOKENS = 150
NUM_TRANSACTIONS = 50000

START_DATE = datetime(2025, 1, 1)

PRICE_INCREMENT = 0.000023
MAX_PRICE = 24.0

def generate_uuid():
    return str(uuid.uuid4())

def random_timestamp():
    return START_DATE + timedelta(days=random.randint(0, 365))

In [ ]:
# from io import StringIO

# from psycopg2 import sql

# # Get the connection string from the environment variable
# conn_string = os.getenv("DATABASE_URL")


# # def load_data(conn_string, df, table_name, *, clear_first=True, truncate_cascade=False):
# #     """Bulk-load a DataFrame into Postgres via COPY.

# #     By default clears the table first with ``TRUNCATE ... RESTART IDENTITY`` so each
# #     run replaces rows (avoids duplicate key errors on reload).

# #     If other tables reference this one, plain ``TRUNCATE`` may fail; set
# #     ``truncate_cascade=True`` to truncate dependent tables too (destructive).

# #     Set ``clear_first=False`` to append without clearing.
# #     """
# #     buf = StringIO()
# #     df.to_csv(buf, index=False, header=True)
# #     buf.seek(0)
# #     with psycopg2.connect(conn_string) as conn:
# #         with conn.cursor() as cur:
# #             cur.execute(sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY CASCADE" ).format(sql.Identifier(table_name)))
# #             #         )
# #             # if clear_first:
# #             #     if truncate_cascade:
# #             #         cur.execute(
# #             #             sql.SQL(
# #             #                 "TRUNCATE TABLE {} RESTART IDENTITY CASCADE"
# #             #             ).format(sql.Identifier(table_name))
# #             #         )
# #             #     else:
# #             #         cur.execute(
# #             #             sql.SQL(
# #             #                 "TRUNCATE TABLE {} RESTART IDENTITY"
# #             #             ).format(sql.Identifier(table_name))
# #             #         )
# #             copy_sql = sql.SQL(
# #                 "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
# #             ).format(sql.Identifier(table_name))
# #             cur.copy_expert(copy_sql, buf)
# #         conn.commit()
# #     print(f"Loaded {len(df)} rows into {table_name!r}")

# def load_data(conn_string, df, table_name, *, clear_first=True, truncate_cascade=False):
#     buf = StringIO()
#     df.to_csv(buf, index=False, header=True)
#     buf.seek(0)

#     with psycopg2.connect(conn_string) as conn:
#         with conn.cursor() as cur:
#             # 1. Properly execute the TRUNCATE
#             if clear_first:
#                 # Add CASCADE if requested
#                 suffix = " CASCADE" if truncate_cascade else ""
#                 truncate_stmt = sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY" + suffix).format(
#                     sql.Identifier(table_name)
#                 )
#                 cur.execute(truncate_stmt) # This sends the command to the DB

#             # 2. Perform the COPY
#             copy_sql = sql.SQL(
#                 "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
#             ).format(sql.Identifier(table_name))
#             cur.copy_expert(copy_sql, buf)
            
#         # 3. Commit the transaction (essential!)
#         conn.commit()
#     print(f"Loaded {len(df)} rows into {table_name!r}")


In [12]:
import os
import psycopg2
from psycopg2 import sql
from io import StringIO
conn_string = os.getenv("DATABASE_URL")

def truncate_table(conn_string, table_name, cascade=False):
    """Forcefully clears the table and resets identities."""
    suffix = " CASCADE" if cascade else ""
    query = sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY" + suffix).format(
        sql.Identifier(table_name)
    )
    
    with psycopg2.connect(conn_string) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
        conn.commit()
    print(f"Table {table_name!r} truncated successfully.")

# def copy_data(conn_string, df, table_name):
#     """Loads DataFrame into the table via COPY."""
#     buf = StringIO()
#     df.to_csv(buf, index=False, header=True)
#     buf.seek(0)
    
#     copy_sql = sql.SQL(
#         "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
#     ).format(sql.Identifier(table_name))
    
#     with psycopg2.connect(conn_string) as conn:
#         with conn.cursor() as cur:
#             cur.copy_expert(copy_sql, buf)
#         conn.commit()
#     print(f"Loaded {len(df)} rows into {table_name!r}")

def copy_data(conn_string, df, table_name):
    buf = StringIO()
    df.to_csv(buf, index=False, header=True)
    buf.seek(0)
    
    # Identify the columns from the DataFrame
    columns = [sql.Identifier(col) for col in df.columns]
    
    # Format: COPY table_name (col1, col2, ...) FROM STDIN ...
    copy_sql = sql.SQL(
        "COPY {} ({}) FROM STDIN WITH (FORMAT csv, HEADER true)"
    ).format(
        sql.Identifier(table_name),
        sql.SQL(', ').join(columns)
    )
    
    with psycopg2.connect(conn_string) as conn:
        with conn.cursor() as cur:
            cur.copy_expert(copy_sql, buf)
        conn.commit()
    print(f"Loaded {len(df)} rows into {table_name!r}")


In [ ]:
# ------------------------
# USERS
# ------------------------
users = []

for _ in range(NUM_USERS):
    user_id = generate_uuid()
    users.append({
        "user_id": user_id,
        "user_role": random.choice(["FAN", "ISSUER", "BOTH"]),
        "display_name": f"user_{random.randint(1000,9999)}",
        "username": f"user_{1000 + _}",
        # "username": f"user_{random.randint(10000,99999)}",
        "referral_id": None,
        "email": f"user{10000 + _}@test.com",
        # "email": f"user{random.randint(10000,99999)}@test.com",
        "email_verified": random.choice([True, False]),
        "email_verification_status": random.choice(["PENDING","VERIFIED"]),
        "email_verification_score": round(random.uniform(50,100),2),
        "country": random.choice(["US","CA","UK"]),
        "zip_code": str(random.randint(10000,99999)),
        "gender": random.choice(["Male","Female","Other"]),
        "time_zone": "UTC",
        # "login_methods_allowed": ["EMAIL_PASSWORD"],
        "password_hash": "hash",
        "passkey_public_key": None,
        "primary_social_platform": None,
        "primary_social_handle": None,
        "mfa_enabled": False,
        "account_status": "ACTIVE",
        "created_at": random_timestamp(),
        "updated_at": None,
        "last_login_at": None,
        "created_ip": "127.0.0.1",
        "created_user_agent": "generator",
        "metadata": None
    })
users_df = pd.DataFrame(users)
users_df.to_csv("./users.csv", index=False)
truncate_table(conn_string, 'users', cascade=True)
copy_data(conn_string, users_df,'users')


Table 'users' truncated successfully.
Loaded 2000 rows into 'users'


In [92]:
# ------------------------
# ISSUERS + VERIFICATION LOGIC
# ------------------------
issuers = []
identity = []
social = []

platforms = ["YOUTUBE", "INSTAGRAM", "X"]

issuer_users = users_df.sample(NUM_ISSUERS).reset_index(drop=True)

PASSED_BOTH = 150

# split remaining 50
remaining = NUM_ISSUERS - PASSED_BOTH

id_only_pass = 30
social_only_pass = 20
none_pass = remaining - id_only_pass - social_only_pass  # 10

for i, row in issuer_users.iterrows():
    issuer_id = generate_uuid()

    # ---------------- STATUS ASSIGNMENT ----------------
    if i < PASSED_BOTH:
        id_status = "PASSED"
        social_status = "PASSED"

    elif i < PASSED_BOTH + id_only_pass:
        id_status = "PASSED"
        social_status = random.choice(["FAILED", "PENDING"])

    elif i < PASSED_BOTH + id_only_pass + social_only_pass:
        id_status = random.choice(["FAILED", "PENDING"])
        social_status = "PASSED"

    else:
        id_status = random.choice(["FAILED", "PENDING"])
        social_status = random.choice(["FAILED", "PENDING"])

    # issuer status derived
    issuer_status = "PASSED" if (id_status == "PASSED" and social_status == "PASSED") else (
        "PENDING" if "PENDING" in [id_status, social_status] else "FAILED"
    )

    # ---------------- ISSUER ----------------
    issuers.append({
        "issuer_id": issuer_id,
        "user_id": row["user_id"],
        "issuer_type": random.choice(["ATHLETE", "CREATOR"]),
        "email": row["email"],
        "username": row["username"],
        "status": issuer_status,
        "level": random.choice(["YOUTH","COLLEGE","PRO"]),
        "country": row["country"],
        "region": "NA",
        "created_at": row["created_at"],
        "updated_at": None,
        "metadata": None
    })

    # ---------------- IDENTITY ----------------
    identity.append({
        "identity_check_id": generate_uuid(),
        "issuer_id": issuer_id,
        "provider": "Persona",
        "status": id_status,
        "level": "basic",
        "alias_confidence": round(random.uniform(70,100),2),
        "opted_in": True,
        "initiated_at": random_timestamp(),
        "completed_at": random_timestamp(),
        "failure_reason": None if id_status == "PASSED" else "check_failed",
        "raw_response": None
    })

    # ---------------- SOCIAL ----------------
    for p in random.sample(platforms, k=1):
        social.append({
            "social_verif_id": generate_uuid(),
            "issuer_id": issuer_id,
            "platform": p,
            "handle": f"{p.lower()}_{random.randint(1000,9999)}",
            "followers_count": random.randint(1000,1000000),
            "verified": True if social_status == "PASSED" else False,
            "initiated_at": random_timestamp(),
            "completed_at": random_timestamp(),
            "attempts": random.randint(1,3),
            "status": social_status,
            "metadata": None
        })

issuers_df = pd.DataFrame(issuers)
identity_df = pd.DataFrame(identity)
social_df = pd.DataFrame(social)

issuers_df.to_csv("./issuers.csv", index=False)
identity_df.to_csv("./identity_verification.csv", index=False)
social_df.to_csv("./social_verification.csv", index=False)

truncate_table(conn_string, 'issuers', cascade=True)
copy_data(conn_string, issuers_df,'issuers')

truncate_table(conn_string, 'identity_verification', cascade=True)
copy_data(conn_string, identity_df,'identity_verification')

truncate_table(conn_string, 'social_verification', cascade=True)
copy_data(conn_string, social_df,'social_verification')

Table 'issuers' truncated successfully.
Loaded 200 rows into 'issuers'
Table 'identity_verification' truncated successfully.
Loaded 200 rows into 'identity_verification'
Table 'social_verification' truncated successfully.
Loaded 200 rows into 'social_verification'


In [107]:
issuers_df.groupby('status').count()

,issuer_id,user_id,issuer_type,email,username,level,country,region,created_at,updated_at,metadata
status,,,,,,,,,,,
FAILED,27,27,27,27,27,27,27,27,27,0,0
PASSED,150,150,150,150,150,150,150,150,150,0,0
PENDING,23,23,23,23,23,23,23,23,23,0,0


In [110]:
d = pd.merge(social_df, issuers_df[['issuer_id', 'status']], on='issuer_id', how='inner')
pd.cut(d[d['status_y']=='PASSED']['followers_count'], bins=[0, 100000, 500000, 1000000]).value_counts()

followers_count
(500000, 1000000]    76
(100000, 500000]     62
(0, 100000]          12
Name: count, dtype: int64

In [115]:
issuer_followers = (
    social_df.groupby("issuer_id")["followers_count"].max()
    .to_dict()
)
issuer_followers

{'01ac40ff-d9f5-44c5-8bec-343b0a577764': 161704,
 '01fc3191-3d92-47bf-93ab-e59076ec3699': 391783,
 '0317ca3a-00ea-4f8f-a042-cefcbb8bbf3c': 741517,
 '05375300-7437-429c-9117-549202d4163b': 392063,
 '06dc2956-265d-4d4b-9be2-0da5ac5a6198': 234709,
 '0a28fb33-7ee3-424f-a251-3326df048a8c': 630073,
 '0c530b7c-bbe9-42d0-9c21-3f7658e81483': 180106,
 '0e342e3a-b7a1-4d73-94a1-22d650ef5a0e': 399395,
 '0e99aba9-6bf5-4aa0-8722-b36bccdc8e17': 875591,
 '0f19ed1c-88c9-480d-a086-859454c480be': 564425,
 '0f23f061-b262-4e56-8ec5-07ab6e09a127': 851702,
 '12757078-cbf9-4cb1-84b4-bbe14e3a0bb2': 653839,
 '127f5dd3-382e-4840-bb09-ac5e549e1332': 716579,
 '129a8e51-f377-4faf-ae81-631c21432e62': 216057,
 '131e865f-5c37-4c8b-985b-e066e0d9cb81': 986810,
 '1469b282-6132-4e58-839e-5adf26e2fc0f': 350914,
 '14a4b425-9408-4db4-803b-31651863b98d': 41664,
 '157925c3-e3fb-4115-9811-504f543cd499': 507921,
 '1651e01a-deef-46e7-b40c-d1773e72185e': 571259,
 '1696df21-0581-49d1-97a0-5c0c0a0870ec': 168008,
 '187bee52-78c1-488c-

In [ ]:
# # ------------------------
# # ISSUERS
# # ------------------------
# issuer_users = users_df.sample(NUM_ISSUERS)
# issuers = []

# for _, row in issuer_users.iterrows():
#     issuer_id = generate_uuid()
#     issuers.append({
#         "issuer_id": issuer_id,
#         "user_id": row["user_id"],
#         "issuer_type": random.choices(["ATHLETE","CREATOR"],weights=[8,1])[0],
#         "email": row["email"],
#         "username": row["username"],
#         "status" : random.choices(["PENDING","PASSED","FAILED"],weights=[2,7.5,0.5])[0],
#         "level": random.choice(["YOUTH","COLLEGE","PRO"]),
#         "country": row["country"],
#         "region": "NA",
#         "created_at": row["created_at"],
#         "updated_at": None,
#         "metadata": None
#     })
# issuers_df = pd.DataFrame(issuers)
# issuers_df.to_csv("./issuers.csv", index=False)
# truncate_table(conn_string, 'issuers', cascade=True)
# copy_data(conn_string, issuers_df,'issuers')



Table 'issuers' truncated successfully.
Loaded 200 rows into 'issuers'


In [53]:
# ------------------------
# ATHLETE PROFILE
# ------------------------
athletes = []
for _, row in issuers_df.iterrows():
    if row["issuer_type"] == "ATHLETE":
        athletes.append({
            "issuer_id": row["issuer_id"],
            "user_id": row["user_id"],
            "sport": random.choice(["Basketball","Soccer","Football"]),
            "team": f"Team_{random.randint(1,20)}",
            "league": "Youth",
            "position_primary": "Forward",
            "position_secondary": None,
            "multi_sport": False,
            "biography": "bio",
            "profile_completion": random.randint(50,100),
            "created_at": random_timestamp(),
            "updated_at": None,
            "metadata": None
        })
athlete_df = pd.DataFrame(athletes)

athlete_df.to_csv("./athletes.csv", index=False)

truncate_table(conn_string, 'athlete_profile', cascade=True)
copy_data(conn_string, athlete_df,'athlete_profile')

Table 'athlete_profile' truncated successfully.
Loaded 200 rows into 'athlete_profile'


In [ ]:
# ------------------------
# POST SIGNUP 
# ------------------------
post_signup = []

for _, row in issuers_df.iterrows():
    post_signup.append({
        "issuer_id": row["issuer_id"],
        "wallet_provisioned": random.choice([True, False]),
        "wallet_address": f"0x{uuid.uuid4().hex[:40]}",
        "wallet_email_sent": True,
        "dashboard_redirect": True,
        "oauth_verified_min2": random.choice([True, False]),
        "verified_at": random_timestamp(),
        "created_at": random_timestamp(),
        "updated_at": None,
        "metadata": None
    })

post_signup_df = pd.DataFrame(post_signup)

post_signup_df.to_csv("./athletes.csv", index=False)

truncate_table(conn_string, 'issuer_post_signup', cascade=True)
copy_data(conn_string, post_signup_df,'issuer_post_signup')

Table 'issuer_post_signup' truncated successfully.
Loaded 200 rows into 'issuer_post_signup'


In [ ]:
# ------------------------
# ISSUER PREFERENCES 
# ------------------------
preferences = []

for _, row in issuers_df.iterrows():
    preferences.append({
        "issuer_id": row["issuer_id"],
        "raise_target_usd": random.randint(10000,1000000), # not needed
        "token_supply_goal": random.randint(100000,1000000), # issuers with higher social followers should have a higher number (loosely)
        "enable_referrals": False,
        "notification_prefs": None,
        "created_at": random_timestamp(),
        "updated_at": None,
        "metadata": None
    })

preferences_df = pd.DataFrame(preferences)

preferences_df.to_csv("./issuer_preferences.csv", index=False)

truncate_table(conn_string, 'issuer_preferences', cascade=True)
copy_data(conn_string, preferences_df,'issuer_preferences')

Table 'issuer_preferences' truncated successfully.
Loaded 200 rows into 'issuer_preferences'


In [108]:
# ------------------------
# TOKENS  (ONLY VERIFIED ISSUERS)
# ------------------------
tokens = []
eligible_issuers = issuers_df[issuers_df["status"] == 'PASSED']

for _, row in eligible_issuers.sample(min(NUM_TOKENS, len(eligible_issuers))).reset_index(drop=True).iterrows():
    tokens.append({
        "token_id": len(tokens) + 1,
        "issuer_id": row["issuer_id"],
        "token_symbol": f"TOKEN_{random.randint(1000,9999)}",
        "initial_supply": random.randint(100000,1000000),
        "current_supply_minted": 0,
        # "numbers_sold": 
        # "current_price":
        "bonding_curve_start_price": 1.000000,
        "bonding_curve_end_price": 24.000000,
        "mint_timestamp": random_timestamp(),
        "paused_sales": False,
        "secondary_trading_enabled": False,
        "total_revenue_raised": 0,
        "created_at": random_timestamp(),
        "updated_at": random_timestamp()
    })

tokens_df = pd.DataFrame(tokens)

truncate_table(conn_string, 'tokens', cascade=True)
copy_data(conn_string, tokens_df,'tokens')

Table 'tokens' truncated successfully.
Loaded 150 rows into 'tokens'


In [109]:
tokens_df

,token_id,issuer_id,token_symbol,initial_supply,current_supply_minted,bonding_curve_start_price,bonding_curve_end_price,mint_timestamp,paused_sales,secondary_trading_enabled,total_revenue_raised,created_at,updated_at
0,1,a9f4aa5c-b255-41d5-a1c2-50db969e4636,TOKEN_4487,926852,0,1.0,24.0,2025-12-10,False,False,0,2025-12-07,2025-01-11
1,2,64c83a99-817f-4bf9-8f6e-18a3f69e2882,TOKEN_6958,125347,0,1.0,24.0,2025-02-21,False,False,0,2025-03-30,2025-05-31
2,3,0e99aba9-6bf5-4aa0-8722-b36bccdc8e17,TOKEN_6744,751375,0,1.0,24.0,2025-05-13,False,False,0,2025-03-04,2025-09-06
3,4,ff1581d7-9072-4b78-8c7d-255e7861d6f7,TOKEN_6519,862005,0,1.0,24.0,2025-10-27,False,False,0,2025-08-06,2025-06-06
4,5,1651e01a-deef-46e7-b40c-d1773e72185e,TOKEN_3653,472993,0,1.0,24.0,2025-10-25,False,False,0,2025-06-21,2025-07-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,146,350643bf-0ccc-4508-be22-7126f54bb633,TOKEN_1068,429753,0,1.0,24.0,2025-02-26,False,False,0,2025-04-10,2025-08-17
146,147,1aebe330-1378-4326-b286-7cb4c003f638,TOKEN_3080,612097,0,1.0,24.0,2025-11-23,False,False,0,2025-01-29,2025-06-14
147,148,e695e634-eaa0-46a8-9337-8a4eeb98c64b,TOKEN_3174,409077,0,1.0,24.0,2025-12-09,False,False,0,2025-11-13,2025-12-29
148,149,7f34039a-0423-4c3b-9de5-56016fab2a60,TOKEN_1348,937401,0,1.0,24.0,2025-11-02,False,False,0,2025-12-17,2025-11-04


In [ ]:
import pandas as pd
import numpy as np
import uuid
import random
from datetime import datetime, timedelta

np.random.seed(42)


# ------------------------
# CONFIG
# ------------------------
NUM_USERS = 2000
NUM_ISSUERS = 200
NUM_TOKENS = 155
NUM_TRANSACTIONS = 50000

START_DATE = datetime(2025, 1, 1)

PRICE_INCREMENT = 0.000023
MAX_PRICE = 24.0

def generate_uuid():
    return str(uuid.uuid4())

def random_timestamp():
    return START_DATE + timedelta(days=random.randint(0, 365))

In [ ]:
# ------------------------
# TRANSACTIONS (PRIMARY ONLY)
# ------------------------
transactions = []
token_prices = {tid: 1.0 for tid in tokens_df["token_id"]}

user_ids = users_df["user_id"].tolist()
token_ids = tokens_df["token_id"].tolist()

for i in range(NUM_TRANSACTIONS):
    token_id = random.choice(token_ids)

    current_price = token_prices[token_id]
    new_price = min(MAX_PRICE, current_price + PRICE_INCREMENT)

    token_prices[token_id] = new_price

    quantity = random.randint(1, 20)

    transactions.append({
        "transaction_id": i + 1,
        "token_id": token_id,
        "buyer_id": random.choice(user_ids),
        "seller_id": None,
        "quantity": quantity,
        #starting price
        "price_per_token": format(new_price, ".6f"),# needs to be new price
        "total_amount_usdc": format(quantity * new_price, ".6f"),
        "transaction_type": "primary",
        "status": "completed",
        "timestamp": random_timestamp()
    })

transactions_df = pd.DataFrame(transactions)

# NEED TRANSACTION SIMULATION ENGINE

In [112]:
def sum_series_formula(n, x):
    # Use // for integer division if n and x are integers
    return (n * x * (n + 1)) // 2

# Example
n = 10
x = 0.000023
print(sum_series_formula(n, x)) # Output: 110

0.0


In [123]:
from decimal import Decimal, getcontext
getcontext().prec = 28

PRICE_INCREMENT = Decimal("0.000023")
START_PRICE = Decimal("1.0")
MAX_PRICE = Decimal("24.0")

# ---------------- BUILD ISSUER FOLLOWER MAP ----------------
# take max followers per issuer from social table
issuer_followers = (
    social_df.groupby("issuer_id")["followers_count"]
    .max()
    .to_dict()
)

# ---------------- HELPER ----------------
def total_cost(n):
    n = Decimal(n)
    first = START_PRICE
    last = START_PRICE + (n - 1) * PRICE_INCREMENT
    return (n / 2) * (first + last)

# ---------------- TRANSACTIONS ----------------
transactions = []
transaction_id = 1

user_ids = users_df["user_id"].tolist()

# build mapping once
issuer_to_user = dict(zip(issuers_df["issuer_id"], issuers_df["user_id"]))

for _, token in tokens_df.iterrows():
    token_id = token["token_id"]
    issuer_id = token["issuer_id"]

    followers = issuer_followers.get(issuer_id, 0)

    # ---------------- RULE 1: NO SALES ----------------
    if followers < 100001:
        continue

    # ---------------- DEMAND FUNCTION ----------------
    # scale sales with followers
    max_tokens_to_sell = int(min(
        token["initial_supply"],
        followers * random.uniform(0.01, 0.05)
    ))

    tokens_sold = 0

    while tokens_sold < max_tokens_to_sell:

        # batch size (simulate transactions)
        batch = random.randint(1, 50)

        if tokens_sold + batch > max_tokens_to_sell:
            batch = max_tokens_to_sell - tokens_sold

        if batch <= 0:
            break

        # ---------------- PRICE CALC ----------------
        start_n = tokens_sold + 1
        end_n = tokens_sold + batch

        # price of next token
        current_price = min(
            MAX_PRICE,
            START_PRICE + (Decimal(start_n - 1) * PRICE_INCREMENT)
        )

        # total cost using arithmetic series
        total_amount = total_cost(end_n) - total_cost(start_n - 1)

        transactions.append({
            "transaction_id": transaction_id,
            "token_id": token_id,
            "buyer_id": random.choice(user_ids),
            "seller_id": issuer_to_user[issuer_id],
            "quantity": batch,
            "price_per_token": format(current_price, ".6f"),
            "total_amount_usdc": format(total_amount, ".6f"),
            "transaction_type": "primary",
            "swap_api_reference": None,
            "original_currency": "USD",
            "status": "completed",
            "reversal_reason": None,
            "timestamp": random_timestamp(),
            "on_chain_tx_hash": None,
            "merkle_proof_hash": None
        })

        tokens_sold += batch
        transaction_id += 1

transactions_df = pd.DataFrame(transactions)

truncate_table(conn_string, 'transactions', cascade=True)
copy_data(conn_string, transactions_df,'transactions')

Table 'transactions' truncated successfully.
Loaded 88727 rows into 'transactions'


In [120]:
truncate_table(conn_string, 'transactions', cascade=True)
copy_data(conn_string, transactions_df,'transactions')

Table 'transactions' truncated successfully.
Loaded 89847 rows into 'transactions'


In [122]:
transactions_df

,transaction_id,token_id,buyer_id,seller_id,quantity,price_per_token,total_amount_usdc,transaction_type,swap_api_reference,original_currency,status,reversal_reason,timestamp,on_chain_tx_hash,merkle_proof_hash
0,1,1,48eb1b3d-65cc-471c-a7d2-0df9398afd8e,a9f4aa5c-b255-41d5-a1c2-50db969e4636,20,1.000000,20.004370,primary,None,USD,completed,None,2025-05-04,None,None
1,2,1,163047d5-4ea6-41e3-9878-55a413d07bd7,a9f4aa5c-b255-41d5-a1c2-50db969e4636,8,1.000460,8.004324,primary,None,USD,completed,None,2025-08-06,None,None
2,3,1,553907c5-8c76-480d-b125-228f248af4dc,a9f4aa5c-b255-41d5-a1c2-50db969e4636,12,1.000644,12.009246,primary,None,USD,completed,None,2025-09-27,None,None
3,4,1,d0936ede-36a2-406a-b4f7-2afa2c883d20,a9f4aa5c-b255-41d5-a1c2-50db969e4636,49,1.000920,49.072128,primary,None,USD,completed,None,2025-04-27,None,None
4,5,1,f793dab7-d0dd-4eb0-a2c0-17dfad7ec522,a9f4aa5c-b255-41d5-a1c2-50db969e4636,40,1.002047,40.099820,primary,None,USD,completed,None,2025-11-19,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88199,88200,150,292730a3-93a0-41f8-a2ff-1f30807d413f,3015aa67-4d3c-48a1-bb3b-9ddd9a8c7cb6,35,1.164473,40.770240,primary,None,USD,completed,None,2025-03-29,None,None
88200,88201,150,29d9b239-1ae5-4d06-bd80-537817735ade,3015aa67-4d3c-48a1-bb3b-9ddd9a8c7cb6,43,1.165278,50.127723,primary,None,USD,completed,None,2025-07-25,None,None
88201,88202,150,978ad204-8c8d-4dfa-acff-8386ae721ead,3015aa67-4d3c-48a1-bb3b-9ddd9a8c7cb6,47,1.166267,54.839412,primary,None,USD,completed,None,2025-09-14,None,None
88202,88203,150,17b65732-614a-4b08-abed-a99e89e2c15f,3015aa67-4d3c-48a1-bb3b-9ddd9a8c7cb6,14,1.167348,16.344965,primary,None,USD,completed,None,2025-05-20,None,None


In [ ]:
I need two more tables. 
First a wallet table that shows assets of a user (crypto (ETH), tokens, USDC. 
Second a credit table so when buyer buys a token issuer gets credited an amount. This table will have same number of transactions, based on trasanactions table. Make sure wallet tables USDC for issuers is equal or bigger than the total amount sold of their token (sum of total_amount_usdc per token - each token belongs to an 